# 04 — Classical Ensemble (V2 Enriched Features)

Train RF v2, XGBoost, SVM, and Logistic Regression on 31 enriched features extracted from the 3-channel motor view (C3, Cz, C4).

Uses V2 data (300 µV PTP threshold → ~4000 epochs vs ~2500 in V1).

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../scripts/run_04_classical_ensemble.py'],
    capture_output=False,
)
result.returncode

In [ ]:
# Evaluate all V2 classical models on the held-out test split and persist metrics.
# Phase C originally reserved test evaluation for the ensemble step; the Streamlit
# training summary needs these per-model test F1 values for review.
import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import sys
sys.path.insert(0, "..")

from src.features_v2 import extract_features_v2
from src.splits import load_split, split_indices_by_subject

ROOT = Path("..").resolve()
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
MODELS = ROOT / "models"

arr = np.load(PROCESSED / "v2_epochs_w0.0-4.0.npz", allow_pickle=False)
X_3ch = arr["X_3ch"]
y = arr["y"]
subject_ids = arr["subject_ids"]
sfreq = float(arr["sfreq"])

split = load_split(PROCESSED / "splits.json")
_, _, test_idx = split_indices_by_subject(subject_ids, split)

features = extract_features_v2(X_3ch, sfreq)
X_test = features[test_idx]
y_test = y[test_idx]

results_path = REPORTS / "v2_classical_results.json"
report = json.loads(results_path.read_text())
entries = {entry["name"]: entry for entry in report["models"]}

for name in ["v2_rf", "v2_xgb", "v2_svm", "v2_lr"]:
    model = joblib.load(MODELS / f"{name}.joblib")
    pred = model.predict(X_test)
    acc = float(accuracy_score(y_test, pred))
    macro_f1 = float(f1_score(y_test, pred, average="macro"))
    cm = confusion_matrix(y_test, pred, labels=[0, 1]).tolist()

    entries[name]["test_accuracy"] = acc
    entries[name]["test_macro_f1"] = macro_f1
    entries[name]["test_confusion_matrix"] = cm
    print(f"{name}: accuracy={acc:.6f} macro_f1={macro_f1:.6f} cm={cm}")

results_path.write_text(json.dumps(report, indent=2))